In [ ]:
%pip install -r requirements.txt

import pandas as pd
import torch
import os
from dotenv import load_dotenv
from src.data import get_test_queries
from src.metrics import calculate_recall_at_k
from src.pipeline import CandidateGenerationPipeline
import pymorphy3

load_dotenv()
hf_token = os.environ.get("HF_TOKEN")

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [2]:
df_train = pd.read_parquet('data/train.parquet')
df_items = pd.read_parquet('data/benchmark_items.parquet')

valid_item_ids = set(df_items["item_id"].unique())
df_train_valid = df_train[df_train["item_id"].isin(valid_item_ids)].copy()
ground_truth, val_queries_df = get_test_queries(df_train_valid, n=1500)

In [3]:
morph = pymorphy3.MorphAnalyzer()

class PymorphyWrapper:
    def stem(self, word):
        return morph.parse(word)[0].normal_form

In [ ]:
params_description_max_length = 500
sentence_transformer_model_name = 'intfloat/multilingual-e5-base'
batch_size=64

pipeline = CandidateGenerationPipeline(
    df_items,
    params_description_max_length=params_description_max_length,
    stemmer=PymorphyWrapper(),
    sentence_transformer_model_name=sentence_transformer_model_name,
    hf_token=hf_token,
    device=device,
    cache_dir='cache',
    batch_size=batch_size
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9560.57it/s]


## Валидационный запуск на тренировочных данных

In [ ]:
# раскомментируйте строки ниже, чтобы запустить валидационный прогон по данным

hidden_phone_weight = 0.9
forbidden_msg_weight = 0.9
review_count_weight = 0.2
title_weight = 0.5
bm25_weight = 0.3
location_weight = 3.5

# val_predictions = pipeline.predict_batch(
#     val_queries_df,
#     top_n=50,
#     title_weight=title_weight,
#     bm25_weight=bm25_weight,
#     location_weight=location_weight,
#     hidden_phone_weight=hidden_phone_weight,
#     forbidden_msg_weight=forbidden_msg_weight,
#     review_count_weight=review_count_weight,
#     norm='min_max'
# )
# recall_score = calculate_recall_at_k(predictions=val_predictions, ground_truth=ground_truth)
# print('Recall score:', recall_score)

Predicting: 100%|██████████| 1500/1500 [15:21<00:00,  1.63it/s]

Recall score: 0.8068608187134504


## Запуск на тестовых данных

In [ ]:
df_queries = pd.read_parquet('data/benchmark_queries.parquet')
predictions = pipeline.predict_batch(
    df_queries,
    top_n=50,
    query_id_col='query_id',
    title_weight=title_weight,
    bm25_weight=bm25_weight,
    location_weight=location_weight,
    hidden_phone_weight=hidden_phone_weight,
    forbidden_msg_weight=forbidden_msg_weight,
    review_count_weight=review_count_weight,
    norm='min_max'
)

Predicting: 100%|██████████| 2452/2452 [20:35<00:00,  1.98it/s]


In [ ]:
# сохранение предсказаний в csv
from src.submit import save_csv_answer
save_csv_answer(predictions, 'answer.csv')